# qbank — Generador de preguntas de opción múltiple

`qbank` genera variantes de preguntas cuyas respuestas correctas se determinan automáticamente mediante lógica proposicional.

Este notebook recorre las principales funcionalidades. Ejecuta las celdas en orden con **Shift+Enter**.

**Contenidos:**
1. [Bloque básico: `ProblemaTipo`](#1)
2. [Revisión del banco: `ProblemaTipoProfe`](#2)
3. [Preguntas paramétricas con `setup`](#3)
4. [Preguntas de verdadero/falso: `ProblemaVF`](#4)
5. [Persistencia en JSON](#5)
6. [Banco de múltiples problemas](#6)
7. [Conversiones entre formatos](#7)
8. [Editor visual](#8)
9. [Preguntas multi-parte](#9)
10. [Exportar a AMC (LaTeX)](#10)
11. [Exportar a Moodle (vía LaTeX)](#11)

In [ ]:
from qbank import *
import itertools, random

<a id="1"></a>
## 1. Bloque básico: `ProblemaTipo`

Un `ProblemaTipo` es una lista de componentes:
- **Cadenas de texto**: aparecen en todas las variantes.
- **Listas de `Supuesto`**: hipótesis alternativas; se elige una por variante y se añade a las hipótesis activas.
- **Listas de `Cuestion`**: ítems de respuesta; se elige uno por variante y se evalúa frente a las hipótesis activas.

**Operadores lógicos** (variables con `v('nombre')`):

| Operador | Significado |
|----------|-------------|
| `v('A')` | variable proposicional A |
| `-v('A')` | negación de A |
| `v('A') & v('B')` | A y B |
| `v('A') \| v('B')` | A o B |
| `v('A') >> v('B')` | si A entonces B |
| `v('A') ** v('B')` | A si y solo si B (equivalencia) |
| `alguno(v('A'), v('B'), v('C'))` | alguna de la lista es verdadera |
| `unoDe(v('A'), v('B'), v('C'))` | exactamente una de la lista es verdadera |
| `True` / `False` | siempre verdadero / siempre falso |

In [ ]:
componentes_p = [
    "Dado que ",
    [
        Supuesto("$\\mathcal{A}$ es verdadero, ", v('A')),
        Supuesto("$\\mathcal{B}$ es verdadero, ", v('B')),
    ],
    "indique qué afirmación es correcta: ",
    [
        Cuestion("$\\mathcal{A}$ es verdadero.", v('A')),
        Cuestion("$\\mathcal{B}$ es verdadero.", v('B')),
    ],
]

p = ProblemaTipo(componentes_p)

for etiqueta, enunciado, cuestiones in p:
    print(f"── Variante {etiqueta} ──")
    print(f"   {enunciado}")
    for texto, correcto, activa, exp in cuestiones:
        print(f"   {'✓' if correcto else '✗'} {texto}")
    print()

<a id="2"></a>
## 2. Revisión del banco: `ProblemaTipoProfe`

`ProblemaTipoProfe` toma la misma lista de componentes que `ProblemaTipo` pero **no descarta
ninguna variante**: muestra todas las combinaciones posibles, incluyendo las inconsistentes.
Es la herramienta adecuada para revisar el banco antes de exportarlo.

Nótese que recibe la lista de componentes (`p.e`), no el objeto `ProblemaTipo`.

In [ ]:
for etiqueta, enunciado, cuestiones in ProblemaTipoProfe(p.e):
    print(f"── Variante {etiqueta} ──")
    print(f"   {enunciado}")
    for texto, correcto, activa, exp in cuestiones:
        marca = '✓' if correcto is True else '✗'
        print(f"   {marca} {texto}")
    print()

<a id="3"></a>
## 3. Preguntas paramétricas con `setup`

El parámetro `setup` es una función sin argumentos que devuelve un diccionario de variables.
Se llama en cada variante, de modo que los valores numéricos o simbólicos cambian en cada llamada.
Los textos de los componentes admiten `@variable` para interpolación de esos valores.

Las semánticas pueden ser lambdas que reciben el diccionario `ns` generado por `setup`,
lo que permite que la veracidad de un ítem dependa de los valores numéricos de la variante.

In [ ]:
def numeros():
    a = random.randint(1, 9)
    b = random.randint(1, 9)
    return {'a': a, 'b': b, 'suma': a + b}

p_param = ProblemaTipo(
    [
        "Sean $a = @a$ y $b = @b$. ",
        [
            Cuestion("$a + b = @suma$", True),
            Cuestion("$a + b > 10$",   lambda ns: ns['a'] + ns['b'] > 10),
            Cuestion("$a = b$",        lambda ns: ns['a'] == ns['b']),
        ],
    ],
    setup=numeros,
)

for etiqueta, enunciado, cuestiones in itertools.islice(p_param, 4):
    print(f"── Variante {etiqueta} ──  {enunciado}")
    for texto, correcto, activa, exp in cuestiones:
        print(f"   {'✓' if correcto else '✗'} {texto}")
    print()

<a id="4"></a>
## 4. Preguntas de verdadero/falso: `ProblemaVF`

`ProblemaVF` genera variantes tomando aleatoriamente `NumPreguntas` ítems de un banco fijo
de cuestiones verdadero/falso. No usa lógica proposicional: la respuesta de cada ítem
es un booleano fijo.

In [ ]:
enunciado_vf = "Indique cuáles de las siguientes afirmaciones son verdaderas:"

banco_vf = [
    ("La derivada de $x^2$ es $2x$.",         True),
    ("La integral de $2x$ es $x^2 + C$.",      True),
    ("$\\sin^2(x) + \\cos^2(x) = 2$.",         False),
    ("El número $e$ es irracional.",            True),
    ("$\\ln(1) = 1$.",                          False),
    ("$\\sqrt{2}$ es irracional.",              True),
]

p_vf = ProblemaVF(enunciado_vf, banco_vf, NumPreguntas=3)

for etiqueta, enunciado, cuestiones in itertools.islice(p_vf, 3):
    print(f"── Variante {etiqueta} ──")
    for texto, correcto in cuestiones:
        print(f"   {'✓' if correcto else '✗'} {texto}")
    print()

<a id="5"></a>
## 5. Persistencia en JSON

`save_problema` y `load_problema` guardan y recuperan un problema (`ProblemaTipo`
o `ProblemaVF`) en formato JSON sin perder ninguna información,
incluidos los `setup` que fueron definidos como cadenas de código.

In [ ]:
import os
os.makedirs("mis_problemas", exist_ok=True)

save_problema(p, "mis_problemas/ejemplo_sec5.json")
print("Guardado en mis_problemas/ejemplo_sec5.json")

p2 = load_problema("mis_problemas/ejemplo_sec5.json")
print("Cargado correctamente. Variantes:")
for etiqueta, enunciado, cuestiones in p2:
    print(f"  Variante {etiqueta}: {enunciado}")

<a id="6"></a>
## 6. Banco de múltiples problemas

`save_banco` y `load_banco` permiten agrupar varios problemas (de cualquier tipo) en un
único fichero JSON. Es el formato habitual cuando se trabaja con colecciones de ejercicios
para un curso o un examen.

In [ ]:
save_banco([p, p_vf], "mis_problemas/banco_sec6.json")
print("Guardado banco con 2 problemas en mis_problemas/banco_sec6.json")

problemas_cargados = load_banco("mis_problemas/banco_sec6.json")
print(f"Cargados {len(problemas_cargados)} problemas:")
for i, prob in enumerate(problemas_cargados):
    print(f"  Problema {i + 1}: {type(prob).__name__}")

<a id="7"></a>
## 7. Conversiones entre formatos

`qbank` usa el diccionario JSON como formato pivote. Desde cualquier representación
puedes obtener cualquier otra:

```
Lista de listas Python
      ↕  problema_to_dict() / problema_from_dict()
   Dict JSON  ←→  fichero .json  (save_problema / load_problema)
      ↓  problema_to_python()
  Código Python editable (.py)
```

In [ ]:
import json

# ── 1. ProblemaTipo → dict JSON ───────────────────────────────────────────────
d = problema_to_dict(p)
print("── Dict JSON ──")
print(json.dumps(d, ensure_ascii=False, indent=2))
print()

In [ ]:
# ── 2. Dict JSON → ProblemaTipo ───────────────────────────────────────────────
p_desde_dict = problema_from_dict(d)
print("── ProblemaTipo recuperado del dict ──")
for etiqueta, enunciado, cuestiones in p_desde_dict:
    print(f"  Variante {etiqueta}: {enunciado}")
    for texto, correcto, _, _ in cuestiones:
        print(f"    {'✓' if correcto else '✗'} {texto}")
print()

In [ ]:
# ── 3. ProblemaTipo → código Python editable ──────────────────────────────────
print("── Código Python generado por problema_to_python() ──")
print(problema_to_python(p))

# Guardar en fichero
save_problema_py(p, "mis_problemas/ejemplo_sec7.py")
print("Guardado en mis_problemas/ejemplo_sec7.py")

<a id="8"></a>
## 8. Editor visual

`ProblemaTipoEditor` es un formulario interactivo para diseñar problemas sin escribir
la lista de listas a mano. Requiere `ipywidgets`.

| Botón | Acción |
|-------|--------|
| **▶ Preview** | muestra las primeras *n* variantes generadas |
| **💾 Guardar** | escribe el JSON en el fichero indicado |
| **📂 Cargar** | carga el JSON del fichero indicado |
| **{ } JSON** | muestra el JSON en el área de salida |
| **⬇ Descargar** | descarga el JSON directamente al navegador |

### 8.1 Editor vacío

##

In [ ]:
from qbank import ProblemaTipoEditor

editor = ProblemaTipoEditor()

> **Nota**: si el editor muestra texto en lugar del formulario (`VBox(children=...`),
> recarga la página y vuelve a ejecutar las celdas desde el principio.

### 8.2 Editor cargando un ejemplo con `setup`

El editor también puede cargar un problema existente desde un fichero JSON.
A continuación se carga `numeros.json`, un ejemplo con `setup` paramétrico:
el campo **Setup** mostrará el código Python que sortea dos números
y calcula su suma y su producto, que luego se interpolan en los textos con `@variable`.

Cada supuesto (`$a>b$`, `$a<b$`, `$a=b$`, `$a\ne b$`) lleva una **precondición**
ligada a los números sorteados (`lambda ns: ns['a'] > ns['b']`, etc.), de modo que
una variante solo se genera cuando lo que afirma es realmente cierto para esa tirada.
Así el enunciado nunca afirma algo falso sobre los números mostrados. El supuesto
`$a\ne b$` se añade para reducir la probabilidad de que el sorteo descarte a la vez
todos los supuestos.

El problema tiene **dos slots de cuestiones** (de tres opciones cada uno) y **sin
texto entre ellos**: no es una pregunta multi-parte. En cada variante se extrae una
cuestión de cada slot y ambas aparecen **juntas**, como las opciones de una misma
pregunta. Por eso el enunciado pide *«Indique qué afirmaciones son correctas»*
(puede haber más de una opción correcta).

La **previsualización** del editor (botón **▶ Preview**) es de tipo *profe*: para
cada combinación de supuestos muestra **todas** las cuestiones posibles, marcando
cada una como correcta o incorrecta. Cuando el sorteo no cumple la precondición de
un supuesto verás el mensaje informativo *«Supuesto … rechazado por
`lambda ns: …`»* (con el código de la precondición, ya legible).

Puedes modificar el problema en el formulario y guardarlo con **💾 Guardar**
o descargarlo con **⬇ Descargar**.

In [ ]:
editor_numeros = ProblemaTipoEditor('numeros.json')

In [ ]:
# El problema cargado en el editor puede exportarse a código Python editable.
# `to_problema()` devuelve el ProblemaTipo con el estado actual del formulario.
print(problema_to_python(editor_numeros.to_problema()))

<a id="9"></a>
## 9. Preguntas multi-parte

`ProblemaTipo` admite ejercicios con varias partes. La lista plana de componentes
determina la estructura: una transición de un bloque de cuestiones a un bloque de
texto o supuestos abre una parte nueva. El método `por_partes()` devuelve
`(etiqueta, [(enunciado, cuestiones), …])` con un par por parte.

`ProblemaMultiParte` y `SubPregunta` siguen disponibles pero están **deprecados**:
devuelven un `ProblemaTipo` equivalente emitiendo un `DeprecationWarning`.

### 9.1 Ejemplo de álgebra lineal

El enunciado introduce una matriz cuadrada bajo dos hipótesis alternativas.
Las dos partes evalúan propiedades independientes de la matriz.

In [ ]:
# v('SP') representa el supuesto «A es simétrica definida positiva»
p_matriz = ProblemaTipo([
    "Sea $A$ una matriz cuadrada ",
    [
        Supuesto("simétrica y definida positiva. ", v('SP')),
        Supuesto(
            "con un autovalor cero cuya multiplicidad "
            "algebraica supera a la geométrica. ",
            -v('SP')),
    ],
    "Con toda seguridad, $A$...",
    [
        Cuestion("es diagonalizable.",
                 v('SP'),
                 exp="Las matrices s.d.p. son siempre diagonalizables "
                     "(teorema espectral)."),
        Cuestion("no es diagonalizable.",
                 -v('SP'),
                 exp="Si mult. algebraica > geométrica, no es diagonalizable."),
    ],
    "En cuanto a $A^T$, con toda seguridad...",
    [
        Cuestion("es invertible.",
                 v('SP'),
                 exp="$A$ s.d.p. implica $A^T = A$ invertible."),
        Cuestion("es singular.",
                 -v('SP'),
                 exp="Autovalor cero $\\Rightarrow$ $\\det(A)=0$ $\\Rightarrow$ "
                     "$\\det(A^T)=0$."),
    ],
])

for etiqueta, partes in p_matriz.por_partes():
    print(f"── Variante {etiqueta} ──")
    for enunciado, cuestiones in partes:
        print(f"   {enunciado}")
        for texto, correcto, _, exp in cuestiones:
            print(f"     {'✓' if correcto else '✗'} {texto}")
            if exp:
                print(f"       ({exp})")
    print()

### 9.2 `ProblemaTipo` multiparte con `setup` numérico

El `setup` también funciona con `ProblemaTipo` multiparte. En este ejemplo se generan
tres enteros aleatorios y cada parte evalúa un cálculo distinto entre ellos.
Las semánticas son lambdas que acceden a los valores numéricos generados por `setup`.

In [ ]:
def tres_enteros():
    a = random.randint(2, 9)
    b = random.randint(2, 9)
    c = random.randint(2, 9)
    return {'a': a, 'b': b, 'c': c,
            'ab': a * b, 'bc': b * c, 'suma': a + b + c}

p_numeros_multi = ProblemaTipo([
    "Sean $a = @a$, $b = @b$ y $c = @c$ números naturales.",
    "El producto $a \\cdot b$...",
    [
        Cuestion("vale $@ab$.",
                 True),
        Cuestion("es mayor que $b \\cdot c = @bc$.",
                 lambda ns: ns['ab'] > ns['bc']),
        Cuestion("es menor o igual que $b \\cdot c = @bc$.",
                 lambda ns: ns['ab'] <= ns['bc']),
    ],
    "La suma $a + b + c$...",
    [
        Cuestion("vale $@suma$.",
                 True),
        Cuestion("es mayor que $a \\cdot b = @ab$.",
                 lambda ns: ns['suma'] > ns['ab']),
        Cuestion("es menor o igual que $a \\cdot b = @ab$.",
                 lambda ns: ns['suma'] <= ns['ab']),
    ],
], setup=tres_enteros)

for etiqueta, partes in itertools.islice(p_numeros_multi.por_partes(), 3):
    print(f"── Variante {etiqueta} ──")
    for enunciado, cuestiones in partes:
        print(f"   {enunciado}")
        for texto, correcto, _, _ in cuestiones:
            print(f"     {'✓' if correcto else '✗'} {texto}")
    print()

<a id="10"></a>
## 10. Exportar a AMC (LaTeX)

Las funciones `AMC*` generan el código LaTeX para **Auto Multiple Choice**.
El fichero resultante se incluye en el documento AMC con `\input{fichero.tex}`.

### 10.1 Pregunta estándar con comodín (`AMClastCh`)

Se exporta el ejemplo paramétrico `numeros.json` (el mismo de la sección 8.2).
La variante `AMClastCh` añade, tras las opciones, una **cuestión comodín**
(`\lastchoices` + *«Ninguna de las anteriores»*) que resulta correcta
únicamente cuando **todas** las demás opciones de la variante son falsas.

In [ ]:
os.makedirs("exportaciones", exist_ok=True)

# Ejemplo paramétrico con setup (el mismo de la sección 8.2)
p_numeros = load_problema("numeros.json")

with open("exportaciones/numeros_amc.tex", "w") as f:
    for etiqueta, enunciado, cuestiones in p_numeros:
        f.write(AMClastCh("Numeros", etiqueta, enunciado, cuestiones))

with open("exportaciones/numeros_amc.tex") as f:
    print(f.read())

### 10.2 Pregunta multi-parte (`AMC_multipart`)

Genera un único `\begin{questionmult}` con un bloque `\begin{choices}` por parte.
Se usa aquí el ejemplo de álgebra lineal (`p_matriz`).

In [ ]:
with open("exportaciones/matriz_amc.tex", "w") as f:
    for etiqueta, partes in p_matriz.por_partes():
        f.write(AMC_multipart("AlgebraLineal", etiqueta, "", partes))

with open("exportaciones/matriz_amc.tex") as f:
    print(f.read())

<a id="11"></a>
## 11. Exportar a Moodle (vía LaTeX)

Las funciones `Quiz*` generan un fichero `.tex` que, al compilarlo con `xelatex`,
produce un `.xml` importable en Moodle.

```bash
xelatex MiCuestionario.tex   # genera MiCuestionario.xml
```

Importar en Moodle: **Banco de preguntas → Importar → Formato Moodle XML**.

### 11.1 Pregunta estándar (`QuizMoodleLastCh`)

In [ ]:
QuizMoodleLastCh("MiCuestionario", "exportaciones/", p)
print("Generado exportaciones/MiCuestionario.tex")

with open("exportaciones/MiCuestionario.tex") as f:
    print(f.read()[:800], "...")

### 11.2 Pregunta cloze multi-parte (`QuizClozeMulti`)

Genera preguntas `\begin{cloze}` con sub-bloques `\begin{multi}` incrustados,
una por cada parte. Se usa aquí el ejemplo de álgebra lineal (`p_matriz`).

In [ ]:
QuizClozeMulti("AlgebraLineal", "exportaciones/", p_matriz)
print("Generado exportaciones/AlgebraLineal.tex")

with open("exportaciones/AlgebraLineal.tex") as f:
    print(f.read()[:800], "...")

---
## Siguiente paso

Consulta el **Manual completo** en `Manual.org` para más detalles sobre:
- Bancos masivos con `setup` paramétrico (dos patrones de exportación)
- Opciones de exportación AMC (`AMCmc`, `AMClastCh`, `AMCmcProfe`, …)
- Cómo usar `ProblemaTipoEditor` para construir y exportar problemas sin escribir código